    Чтение и поиск параметров в конфигурационном файле MetaTrader 5 SERVER; ноходящийся в zip архиве.

In [1]:
# --- SETTINGS ---

# Скрипт работает в двух режимах: тестовом и реальном.
IS_TEST_MODE = True  # Смени на False для работы с реальными данными

# Предположительно, универсальный блок, для контроля наличия и создания директорий для работы скрипта, а также для добавления в sys.path пути к библиотекам

from pathlib import Path
import datetime
import sys
import os
import io
import json
import zipfile
import tempfile
import pandas as pd

# Функция для проверки существования директории и её создания при отсутствии <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
def ensure_directory(path, description="", name=""):
    if not os.path.isdir(path):
        os.makedirs(path)
        print(f"❗📁 [{name}] не найден, создан новый каталог: {os.path.abspath(path)}")
    else: print(f"    📁 [{name}]; {description}: {os.path.abspath(path)}")
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

print       ("Заведомо существующие директории:")
file_dir = os.getcwd()                                      # Определяем путь к текущему файлу (где выполняется код)
ensure_directory(file_dir, "Путь к директории с ipynb файлами", "file_dir")

sub_project_dir = Path(file_dir).parent
ensure_directory(sub_project_dir, "Путь к директории СубПроекта", "sub_project_dir")

project_dir = Path(sub_project_dir).parent                  # Переход в верхнюю директорию проекта (fc_to_mt5_migrations/own_platform)
ensure_directory(project_dir, "Путь к директории Проекта", "project_dir")

parent_dir = Path(project_dir).parent                       # Переход на уровень выше (fc_to_mt5_migrations)
ensure_directory(parent_dir, "Путь к директории для доступа к библиотекам", "parent_dir")

print       ("\n Директории СубПроекта:")
print       (" Директории Исходных Данных:")
input_log_data = os.path.join(project_dir, sub_project_dir, "input_data", 'input_log_data')
ensure_directory(input_log_data, "Путь к каталогу с логами исходных файлов", "input_log_data")

input_temp_data = os.path.join(project_dir, sub_project_dir, "input_data", 'input_temp_data')
ensure_directory(input_temp_data, "Путь к каталогу с временными файлами", "input_temp_data")

input_samples_data = os.path.join(project_dir, sub_project_dir, "input_data", 'input_samples')
ensure_directory(input_samples_data, "Путь к каталогу с примерами исходных данных данных, для запуска в тестовом режиме", "input_samples_data")

print       (" Директории Данных полученных в процессе работы скрипта:")
output_log_data = os.path.join(project_dir, sub_project_dir, "output_data", 'output_log_data')
ensure_directory(output_log_data, "Путь к каталогу с логами выходных файлов", "output_log_data")

output_temp_data = os.path.join(project_dir, sub_project_dir, "output_data", 'output_temp_data')
ensure_directory(output_temp_data, "Путь к каталогу с временными файлами", "output_temp_data")

directory_data_set = os.path.join(project_dir, 'data_set')
ensure_directory(directory_data_set, "Путь к каталогу с Файлами Постоянных Конфигураций", "directory_data_set")

libraries_path = os.path.join(parent_dir, "libraries_py")                           # Формируем путь к libraries_py каталогу с библиотеками *.py
ensure_directory(libraries_path, "Путь к директории с библиотеками", "libraries_path")
sys.path.append(libraries_path)                                                     # sys.path — это список путей, где Python ищет модули при import module_name.

print("\n Проверка наличия каталога с библиотеками в sys.path:")
if libraries_path in sys.path: print(f"✅ Каталог {libraries_path} успешно добавлен в sys.path")
else: print(f"❌ Ошибка: {libraries_path} не найден в sys.path")

Заведомо существующие директории:
    📁 [file_dir]; Путь к директории с ipynb файлами: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\Analysis_MT5_configuration\ipynb_files
    📁 [sub_project_dir]; Путь к директории СубПроекта: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\Analysis_MT5_configuration
    📁 [project_dir]; Путь к директории Проекта: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform
    📁 [parent_dir]; Путь к директории для доступа к библиотекам: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations

 Директории СубПроекта:
 Директории Исходных Данных:
    📁 [input_log_data]; Путь к каталогу с логами исходных файлов: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\Analysis_MT5_configuration\input_data\input_log_data
    📁 [input_temp_data]; Путь к каталогу с временными файлами: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\Analysis_MT5

In [2]:
# Динамически импорт необходимых функций <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
file_imports = "dynamic_import_functions.py"                                    # Библиотека для динамического импорта
file_imports_path = os.path.join(libraries_path, file_imports)
if os.path.exists(file_imports_path):
    import importlib
    importlib.invalidate_caches()                                               # Сбрасываем кэш перед импортом
    from dynamic_import_functions import import_functions, print_import_function_info
    print(f"\n ✅ Импорт [{file_imports}] успешен.")
else:
    print(f"\n ERROR: Файл '{file_imports}' не найден по пути {file_imports_path}, импорт не выполнен.\n")

modules_to_import = {                                                           # Формируем словарь, с именами файлов и функциями в них
    "yar_sed_general_lib":
        [libraries_path,
                "pd_set_option",                        # Вывод ДФ
                "df_to_csv",                            # Сохранение ДФ в CSV 
                "CSVLoader",
                "save_data_log_work_file",
                "detect_encoding",
                "collect_keys_by_depth",          # Сбор ключей по уровням вложенности
                "files_in_directory",          # Вывод файлов в директории}
                "save_dict_data_log_work_file_zip",
                "read_json_from_zip_or_file",
                ]
                }
imported = import_functions(modules_to_import)          # Импортируем модули из словаря modules_to_import

print_import_function_info(modules_to_import, imported) # Выводим переменные ожидаемые импортированными функциями 


 ✅ Импорт [dynamic_import_functions.py] успешен.
Импорт из 'yar_sed_general_lib' успешен: ['pd_set_option', 'df_to_csv', 'CSVLoader', 'save_data_log_work_file', 'detect_encoding', 'collect_keys_by_depth', 'files_in_directory', 'save_dict_data_log_work_file_zip', 'read_json_from_zip_or_file']

 Импортированные функции и их параметры:
Функция 'pd_set_option' из модуля 'yar_sed_general_lib' ожидает параметры: name_df: str, df: pandas.core.frame.DataFrame, rows: int = 10, columns: int | None = None, min_rows: int | None = None, width: int = 100
Функция 'df_to_csv' из модуля 'yar_sed_general_lib' ожидает параметры: df, csv_file_path
Функция 'CSVLoader' из модуля 'yar_sed_general_lib' ожидает параметры: file_path, delimiter=';', encoding='utf-8', df_name='dataframe'
Функция 'save_data_log_work_file' из модуля 'yar_sed_general_lib' ожидает параметры: df, file_name, directory_data_temp_files, directory_data_log_files
Функция 'detect_encoding' из модуля 'yar_sed_general_lib' ожидает параметры:

In [3]:
# Скрипт не предполагал запроса файлов конфигураций, хотя можно было расширить функционал и запрашивать файлы с сервера либо из каталога
# с архивными конфигурациями сервера. В данном случае,
# для тестового режима, мы будем использовать файлы из каталога input_samples_data,
# а для реального режима - из каталога input_temp_data.
if IS_TEST_MODE:
    directory_data = input_samples_data
    print(f"\n ✅ Тестовый режим: Используем каталог с примерами исходных данных: {directory_data}")
else:
    directory_data = input_temp_data
    print(f"\n ✅ Реальный режим: Используем каталог с временными данными: {directory_data}")
imported["files_in_directory"](directory_data)  # Выводим файлы в директории


 ✅ Тестовый режим: Используем каталог с примерами исходных данных: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\Analysis_MT5_configuration\input_data\input_samples
функция [files_in_directory] вывода файлов; operating_mode = 0; 
 0 - только вывод; 1 - только список; 2 - и вывод и список; 
 директория: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\Analysis_MT5_configuration\input_data\input_samples:
  250108 ALL BarclayStoneTest-Demo.zip


In [4]:
# Чтение конфигурационного файла находящегося в ZIP архиве <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
# Формируем пути к json и zip
filename = "250108 ALL BarclayStoneTest-Demo"
filename_extension = filename + ".json"
zip_filename = filename + ".zip"

json_path = os.path.join(directory_data, filename_extension)
zip_path = os.path.join(directory_data, zip_filename)

data = imported["read_json_from_zip_or_file"](zip_path, json_path, filename_extension)

keys_by_level = imported["collect_keys_by_depth"](data)         # 4. Сбор и вывод ключей по уровням вложенности

for level in sorted(keys_by_level.keys()): print(f"Уровень {level}: {sorted(keys_by_level[level])}")

Определение кодировки файла: C:\Users\nigilist\AppData\Local\Temp\tmpuw60gt86.json
Определённая кодировка: utf_8
Уверенность (0.0 — отлично, ближе к 1.0 — плохо): 0.0 

Уровень 0: ['Server']
Уровень 1: ['ConfigAllocations', 'ConfigAutomation', 'ConfigCommon', 'ConfigECNCommon', 'ConfigFeeders', 'ConfigFirewall', 'ConfigGroups', 'ConfigHistorySync', 'ConfigHolidays', 'ConfigManagers', 'ConfigNetwork', 'ConfigPlugins', 'ConfigReports', 'ConfigRouting', 'ConfigSubscriptions', 'ConfigSymbols', 'ConfigTime', 'ConfigVPS', 'ConfigWebService']
Уровень 2: ['Access', 'AccessServer', 'AccountAuto', 'AccountDepositUrl', 'AccountGroups', 'AccountType', 'AccountUrl', 'AccountWithdrawalUrl', 'AccruedInterest', 'Action', 'ActionValueFloat', 'ActionValueInt', 'ActionValueString', 'ActionValueUInt', 'Actions', 'Adapter', 'Adapters', 'Address', 'AddressIPv6', 'Addresses', 'AgreementURL', 'Agreements', 'AttemptsSleep', 'AuthMode', 'AuthOTPMode', 'AuthPasswordMin', 'BackupServer', 'Basis', 'Binds', 'CFI', 

In [5]:
# Поиск путей с определенной группой в JSON файле <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
target_group = "demo\\technical_dev\\migration_zero_usd"                                    # Задаем целевую группу для поиска
#target_group = "demo\\bv\\migration_zero_usd"
paths = []

def find_paths_with_target_group(obj):
    if isinstance(obj, dict):
        if obj.get("Group") == target_group and "Symbols" in obj:
            for symbol in obj["Symbols"]:
                if isinstance(symbol, dict) and "Path" in symbol:
                    paths.append(symbol["Path"])
        # Рекурсивно обходим вложенные словари
        for value in obj.values():
            find_paths_with_target_group(value)
    elif isinstance(obj, list):
        for item in obj:
            find_paths_with_target_group(item)

find_paths_with_target_group(data)

if paths:                                                                                   # Вывод результатов
    print(f"🔎Клиентской группе [{target_group}]  доступны следующие торговые инструменты:")
    for path in paths:
        print(path)
else:   print(f"❌ Не найдено записей с Group = '{target_group}'")


🔎Клиентской группе [demo\technical_dev\migration_zero_usd]  доступны следующие торговые инструменты:
FOREX\*
STOCKS\*
CRYPTO\*
Spot Metals\*
Spot Energy\*
Spot Energy\NG
Spot Commodities\*
Spot Indices STD\*
custom\Bitwise_LTD


In [6]:
# Удалим * и \ только в конце строки
prefixes = [g.rstrip("*").rstrip("\\").strip() for g in paths if g.endswith("*")]
print(f"prefixes = {prefixes}")  # ['FOREX', ...]

symbols_x = []
zero_level = next(iter(data.values()))

config_symbols_list = None
for item in zero_level:
    if isinstance(item, dict) and "ConfigSymbols" in item:
        config_symbols_list = item["ConfigSymbols"]
        break

if config_symbols_list is None: print("❌ Ключ 'ConfigSymbols' не найден.")
else: print(f"✅ Найдено {len(config_symbols_list)} записей в 'ConfigSymbols'.")

rows = []                                                                           # Сюда будем собирать данные для ДФ
for item in config_symbols_list:
    path = item.get("Path", "").strip()
    symbol = item.get("Symbol", "")
    base_path = path.split("\\", 1)[0].strip()

    #print(f"Путь: {path!r}, Базовый путь: {base_path!r}")
    if base_path in prefixes:
        #print(f"✔️  Совпадение: {base_path} ∈ prefixes")
        symbols_x.append(symbol)
        rows.append({
            "Путь": path,
            "Базовый путь": base_path,
            "symbol": symbol
        })
print(f"Найденные символы: {(symbols_x)}")                                          # Вывод найденных символов
df_symbols = pd.DataFrame(rows)
display(df_symbols)
df_symbols["Базовый путь"].value_counts()


prefixes = ['FOREX', 'STOCKS', 'CRYPTO', 'Spot Metals', 'Spot Energy', 'Spot Commodities', 'Spot Indices STD']
✅ Найдено 9923 записей в 'ConfigSymbols'.
Найденные символы: ['AUDCAD', 'AUDCHF', 'AUDCNH', 'AUDDKK', 'AUDEUR', 'AUDGBP', 'AUDHUF', 'AUDINR', 'AUDJPY', 'AUDMEX', 'AUDNOK', 'AUDNZD', 'AUDPLN', 'AUDRUB', 'AUDSEK', 'AUDSGD', 'AUDTRL', 'AUDZAR', 'CADAUD', 'CADCHF', 'CADCNH', 'CADDKK', 'CADEUR', 'CADGBP', 'CADHUF', 'CADINR', 'CADJPY', 'CADMEX', 'CADNOK', 'CADNZD', 'CADPLN', 'CADRUB', 'CADSEK', 'CADSGD', 'CADTRL', 'CADZAR', 'CHFAUD', 'CHFCAD', 'CHFCNH', 'CHFDKK', 'CHFEUR', 'CHFGBP', 'CHFHUF', 'CHFINR', 'CHFJPY', 'CHFMEX', 'CHFNOK', 'CHFNZD', 'CHFPLN', 'CHFRUB', 'CHFSEK', 'CHFSGD', 'CHFTRL', 'CHFZAR', 'CNHCAD', 'CNHCHF', 'CNHEUR', 'CNHGBP', 'CNHJPY', 'CNHMEX', 'CNHRUB', 'CNHTRL', 'CNHZAR', 'CZKCAD', 'CZKCHF', 'CZKEUR', 'CZKGBP', 'CZKJPY', 'CZKNOK', 'CZKSEK', 'DKKAUD', 'DKKCAD', 'DKKCHF', 'DKKEUR', 'DKKGBP', 'DKKJPY', 'DKKNOK', 'DKKNZD', 'DKKSEK', 'EURAUD', 'EURCAD', 'EURCHF', 'EURCNH

,Путь,Базовый путь,symbol
0,FOREX\Fx Cross Rates\AUDCAD,FOREX,AUDCAD
1,FOREX\Fx Cross Rates\AUDCHF,FOREX,AUDCHF
2,FOREX\Fx Cross Rates\AUDCNH,FOREX,AUDCNH
3,FOREX\Fx Cross Rates\AUDDKK,FOREX,AUDDKK
4,FOREX\Fx Cross Rates\AUDEUR,FOREX,AUDEUR
...,...,...,...
7917,STOCKS\CFDs - Stocks UK\RNO.L,STOCKS,RNO.L
7918,STOCKS\CFDs - Stocks IT\UCG.IT,STOCKS,UCG.IT
7919,Spot Metals\SILVER,Spot Metals,SILVER
7920,STOCKS\CFDs - Stocks US\NVST.N,STOCKS,NVST.N


Базовый путь
STOCKS              7285
FOREX                304
CRYPTO               281
Spot Commodities      21
Spot Indices STD      17
Spot Metals            8
Spot Energy            6
Name: count, dtype: int64

In [9]:
pretty_json = json.dumps(data, indent=4, ensure_ascii=False)    # Красивый вывод с табуляцией json
#print(pretty_json)
imported["save_dict_data_log_work_file_zip"](pretty_json, "BarclayStoneTest-Demo", output_temp_data, output_log_data)

Сохранение словаря в рабочую/временную папку и лог-папку: BarclayStoneTest-Demo
Вспомогательная функция для упаковывания словаря/JSON в .zip файл.
💾 Архив со Словарём сохранён в файл: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\Analysis_MT5_configuration\output_data\output_temp_data\BarclayStoneTest-Demo.zip
Вспомогательная функция для упаковывания словаря/JSON в .zip файл.
💾 Архив со Словарём сохранён в файл: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\Analysis_MT5_configuration\output_data\output_log_data\BarclayStoneTest-Demo_20260812_165653.zip
